## Dataset Input

In [1]:

from pathlib import Path

# Locate the existing project root
BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
DATASET_DIR = RAW_DIR / "spacenet8"

# Compatibility aliases for later lab cells
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("SpaceNet 8 directory:", DATASET_DIR)

assert DATASET_DIR.is_dir(), "SpaceNet 8 dataset folder not found."
print("Workspace initialized successfully.")


Project root: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard
Raw data directory: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard\data\raw
Processed data directory: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard\data\processed
SpaceNet 8 directory: C:\Users\Qiha\Documents\GitHub\ai-engineering-flooddashboard\data\raw\spacenet8
Workspace initialized successfully.


In [2]:

from collections import Counter

# Find all extracted files
all_files = sorted(
    file for file in DATASET_DIR.rglob("*")
    if file.is_file()
)

file_types = Counter(
    file.suffix.lower() for file in all_files
)

print("Total files:", len(all_files))
print("\nFile types:")
for ext, count in sorted(file_types.items()):
    print(f"{ext or '[no extension]'}: {count}")

print("\nSample files:")
for file in all_files[:15]:
    print(file.relative_to(DATASET_DIR))

# Verify expected folders
for folder in ["PRE-event", "POST-event", "annotations"]:
    path = DATASET_DIR / folder
    print(f"\n{folder}: {'Found' if path.is_dir() else 'Missing'}")

assert all_files, "No input files found."
print("\nInput verification completed.")


Total files: 689

File types:
.csv: 2
.geojson: 202
.gz: 1
.tif: 484

Sample files:
annotations\0_15_63.geojson
annotations\0_15_65.geojson
annotations\0_15_66.geojson
annotations\0_15_67.geojson
annotations\0_15_68.geojson
annotations\0_15_69.geojson
annotations\0_15_70.geojson
annotations\0_16_63.geojson
annotations\0_16_66.geojson
annotations\0_16_67.geojson
annotations\0_16_68.geojson
annotations\0_16_70.geojson
annotations\0_17_63.geojson
annotations\0_17_66.geojson
annotations\0_17_67.geojson

PRE-event: Found

POST-event: Found

annotations: Found

Input verification completed.


## Count pre-events and post-event images

In [3]:

pre_images = sorted((DATASET_DIR / "PRE-event").rglob("*.tif"))
post_images = sorted((DATASET_DIR / "POST-event").rglob("*.tif"))
annotations = sorted((DATASET_DIR / "annotations").rglob("*.geojson"))

print("PRE-event images:", len(pre_images))
print("POST-event images:", len(post_images))
print("GeoJSON annotations:", len(annotations))

print("\nExample PRE-event image:")
print(pre_images[0].relative_to(DATASET_DIR) if pre_images else "None")

print("\nExample POST-event image:")
print(post_images[0].relative_to(DATASET_DIR) if post_images else "None")

print("\nExample annotation:")
print(annotations[0].relative_to(DATASET_DIR) if annotations else "None")


PRE-event images: 202
POST-event images: 282
GeoJSON annotations: 202

Example PRE-event image:
PRE-event\10500500C4DD7000_0_15_63.tif

Example POST-event image:
POST-event\1040050035DC3B00_0_15_63.tif

Example annotation:
annotations\0_15_63.geojson


## GeoTIFF metadata
Checking image metadata (CRS, resolution, bands and dimensions)

In [4]:

import rasterio

def inspect_geotiff(path):
    with rasterio.open(path) as src:
        return {
            "file": str(path.relative_to(DATASET_DIR)),
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "dtype": src.dtypes,
            "crs": str(src.crs),
            "resolution": src.res,
            "nodata": src.nodata,
        }

if pre_images and post_images:
    pre_info = inspect_geotiff(pre_images[0])
    post_info = inspect_geotiff(post_images[0])

    print("PRE-EVENT IMAGE")
    for key, value in pre_info.items():
        print(f"{key}: {value}")

    print("\nPOST-EVENT IMAGE")
    for key, value in post_info.items():
        print(f"{key}: {value}")


PRE-EVENT IMAGE
file: PRE-event\10500500C4DD7000_0_15_63.tif
width: 1300
height: 1300
bands: 3
dtype: ('uint8', 'uint8', 'uint8')
crs: EPSG:4326
resolution: (3.74298035050102e-06, 3.74298035050102e-06)
nodata: 255.0

POST-EVENT IMAGE
file: POST-event\1040050035DC3B00_0_15_63.tif
width: 1743
height: 1742
bands: 3
dtype: ('uint8', 'uint8', 'uint8')
crs: EPSG:4326
resolution: (2.793760533611708e-06, 2.793760533611708e-06)
nodata: None


## GeoJSON annotation

In [5]:
import json
from collections import Counter

if annotations:
    annotation_path = annotations[0]

    with annotation_path.open("r", encoding="utf-8") as f:
        annotation_data = json.load(f)

    features = annotation_data.get("features", [])

    geometry_types = Counter(
        feature.get("geometry", {}).get("type", "Unknown")
        for feature in features
    )

    print("Annotation file:", annotation_path.name)
    print("GeoJSON type:", annotation_data.get("type"))
    print("Number of features:", len(features))
    print("Geometry types:", dict(geometry_types))

    if features:
        print("\nFirst feature properties:")
        print(json.dumps(features[0].get("properties", {}), indent=2))


Annotation file: 0_15_63.geojson
GeoJSON type: FeatureCollection
Number of features: 20
Geometry types: {'LineString': 7, 'Polygon': 13}

First feature properties:
{
  "id": "way/847402035",
  "building": null,
  "flooded": null,
  "highway": "residential",
  "lanes": "2",
  "surface": "paved",
  "oneway": null,
  "bridge": null,
  "layer": null,
  "tunnel": null,
  "type": null,
  "image_id": "10500500C4DD7000",
  "location": "Germany_Training_Public"
}


## CSV metadata

In [6]:
import pandas as pd

mapping_path = (
    DATASET_DIR /
    "Germany_Training_Public_label_image_mapping.csv"
)

reference_path = (
    DATASET_DIR /
    "Germany_Training_Public_reference.csv"
)

mapping_df = pd.read_csv(mapping_path)
reference_df = pd.read_csv(reference_path)

print("LABEL-IMAGE MAPPING")
print("Shape:", mapping_df.shape)
print("Columns:", mapping_df.columns.tolist())
display(mapping_df.head(3))

print("\nREFERENCE METADATA")
print("Shape:", reference_df.shape)
print("Columns:", reference_df.columns.tolist())
display(reference_df.head(3))


LABEL-IMAGE MAPPING
Shape: (202, 4)
Columns: ['label', 'pre-event image', 'post-event image 1', 'post-event image 2']


,label,pre-event image,post-event image 1,post-event image 2
0,0_41_59.geojson,10500500C4DD7000_0_41_59.tif,10500500E6DD3C00_0_41_59.tif,NaN
1,0_45_64.geojson,10500500C4DD7000_0_45_64.tif,10500500E6DD3C00_0_45_64.tif,NaN
2,0_32_59.geojson,10500500C4DD7000_0_32_59.tif,10500500E6DD3C00_0_32_59.tif,NaN



REFERENCE METADATA
Shape: (9761, 6)
Columns: ['ImageId', 'Object', 'Wkt_Pix', 'Flooded', 'length_m', 'travel_time_s']


,ImageId,Object,Wkt_Pix,Flooded,length_m,travel_time_s
0,10500500C4DD7000_0_44_69,Road,LINESTRING EMPTY,Null,Null,Null
1,10500500C4DD7000_0_37_69,Road,LINESTRING (1300.0000000000637 125.11088048440...,False,9.594785852252468,0.7154308229131224
2,10500500C4DD7000_0_37_69,Road,LINESTRING (1264.8329929537006 119.92324021553...,False,26.11318204209286,1.9471174870326937


## Data inventory

In [7]:
# Check missing metadata values
print("MAPPING CSV — MISSING VALUES")
print(mapping_df.isna().sum())

print("\nREFERENCE CSV — OBJECT TYPES")
print(reference_df["Object"].value_counts(dropna=False))

print("\nREFERENCE CSV — FLOODED LABELS")
print(reference_df["Flooded"].value_counts(dropna=False))

# Check whether mapped files exist
mapping_checks = []

for _, row in mapping_df.iterrows():
    label = DATASET_DIR / "annotations" / str(row["label"])
    pre = DATASET_DIR / "PRE-event" / str(row["pre-event image"])

    post_files = [
        DATASET_DIR / "POST-event" / str(row[column])
        for column in ["post-event image 1", "post-event image 2"]
        if pd.notna(row[column])
    ]

    mapping_checks.append({
        "label_exists": label.is_file(),
        "pre_exists": pre.is_file(),
        "post_exists": all(path.is_file() for path in post_files),
        "post_count": len(post_files)
    })

checks_df = pd.DataFrame(mapping_checks)

print("\nIMAGE-LABEL MAPPING INTEGRITY")
print(checks_df.agg({
    "label_exists": "sum",
    "pre_exists": "sum",
    "post_exists": "sum",
    "post_count": "sum"
}))

print("\nMappings with missing files:")
print(
    (~checks_df[["label_exists", "pre_exists", "post_exists"]])
    .any(axis=1)
    .sum()
)


MAPPING CSV — MISSING VALUES
label                   0
pre-event image         0
post-event image 1      0
post-event image 2    122
dtype: int64

REFERENCE CSV — OBJECT TYPES
Object
Road        5702
Building    4059
Name: count, dtype: int64

REFERENCE CSV — FLOODED LABELS
Flooded
False    7183
True     2498
Null       80
Name: count, dtype: int64

IMAGE-LABEL MAPPING INTEGRITY
label_exists    202
pre_exists      202
post_exists     202
post_count      282
dtype: int64

Mappings with missing files:
0
